# SI — Effect of z defocus on the localisation and the spectral fingerprint

**Makes up** (SI section *Effect of Z Defocus on Extraction of Spectral Signature*):
`exemplar_psfs_grid_488.svg` (beads at five distances from focus), `precision_vs_z.svg`
(localisation and spectral precision against z), `ridge_colour_qe_vs_z_488_small.svg` (relative pixel
QE at −200 to +200 nm) and `ridge_colour_qe_vs_z_488.svg` (−1,000 to +1,000 nm).

**Data:** BioStudies `S-BIAD3210`, `2026_Multicolour_Paper/Ximea/Z_Defocus/PFS_zstack_488`: TetraSpeck
beads in 1 % PVA on the Ximea MC050CG-SY, 488 nm excitation, Perfect Focus System on; 9 fields of view,
each 21 z-steps from −1,000 to +1,000 nm (100 nm apart) × 100 frames (2,100 frames, split by
MicroManager into a main `.ome.tif` and a `_1.ome.tif`), ~42 GB in all; `zlog.csv` is the stage log.

**Method** (`_z_defocus.py`):
1. Fit: spots are detected once per field on the sum of all 2,100 frames (in-focus frames dominate),
   then every frame is fitted at every spot (`fit_QD_data`: STANDARD_DATA Gaussian, 40 px box, false-alarm
   rate 10⁻⁴, no upper PSF width bound so that defocused PSFs are kept).
2. Select: colour-fraction errors < 1 (no other fit cut: defocused PSFs are wide and imprecise by
   design); per localisation ≥ 700 detected photoelectrons and pixel-2 share ≥ 0.5; then beads present
   in ≥ 19 of the 21 z-steps.
3. z = 0 is the z-step with the narrowest mean PSF over all beads (with the PFS on, the beads share
   one focal plane).
4. Per z-step: σ_xy = RMS offset of each localisation from its own bead's mean position at that step;
   σ_colour of pixel p = SD of $\hat A_p$ about its own bead's mean $\hat A_p$ at that step, in units of
   relative pixel QE. It is not divided by the mean (as in Fig. 2d): at 488 nm the TetraSpeck pixel-3
   share is ~0.01, which makes a relative SD meaningless (`rel_sigma_colour_*` in the CSV has it anyway).
5. z = 0 falls at nominal +100 nm with all 9 fields, so the scan covers −1,100 to +900 nm relative to
   focus; the full-range figures use −1,000, −500, 0, +500 and +900 nm.

Changed from the developer notebook (`z_defocus_data/z_defocus_analysis`): no +200 nm shift of z in the
precision plot and PSF grid (z = 0 is the narrowest PSF everywhere); σ_colour is per bead, per pixel
type (it was the spread of all beads' fingerprints about their common mean); no figure titles. The fitter
now stops at ftol and xtol 10⁻⁸ for boxes wider than 25 px (it stopped at 10⁻², far from the minimum: the
2026-11-09 fits put pixel 3 at ~0 for dim beads and their fingerprint depended on the starting guess).

## FAST_MODE
`FAST_MODE = True` (default) downloads and fits one field (`Pos-0-0`, ~4.7 GB, ~10 beads); every figure
is made from it. `FAST_MODE = False` uses all 9 fields (~42 GB).

---
## 0 — Imports

In [ ]:
import sys
import types
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# _biostudies_download.py and _z_defocus.py live one folder up.
sys.path.insert(0, str(Path.cwd().resolve().parent))
import _biostudies_download as bsd
import _z_defocus as zd

from pyS3M import IOFunctions, sCMOSFunctions, SR_Functions, SM_extractionfunctions, PlottingBase, MaskFunctions
from pyS3M.CameraDefaults import get_camera_config

IO = IOFunctions.IO_Functions(camera="ximea")
sCMOS = sCMOSFunctions.sCMOS_Functions()
SupRes_F = SR_Functions.SuperRes_Functions(camera="ximea")
SM_E = SM_extractionfunctions.extract_SMs(camera="ximea")
plotter = PlottingBase.PublicationPlotter(dark_background=False)

---
## 1 — Configuration

In [ ]:
FAST_MODE = True    # True: one field (~4.7 GB). False: all 9 fields (~42 GB).
REFIT = False       # True: fit again even if fits exist (old fits are moved to old_fits/).

REPO_ROOT = Path.cwd().resolve().parents[2]   # notebooks/figures/SI -> repo root
CALIB_DIR = REPO_ROOT / "Camera_Calibrations" / "Ximea_Camera"
OUT_ROOT  = REPO_ROOT / "outputs" / "FigureSI_ZDefocus"   # git-ignored
DATA_DIR  = OUT_ROOT / ("data_fast" if FAST_MODE else "data_full")
FIG_DIR   = OUT_ROOT / ("fast" if FAST_MODE else "full")
FIG_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

calib = dict(
    gain_map=IO.read_tiff(CALIB_DIR / "gain.tif"),
    offset_map=IO.read_tiff(CALIB_DIR / "offset.tif"),
    rqe=IO.read_tiff(CALIB_DIR / "rqe.tif"),
    read_noise=IO.read_tiff(CALIB_DIR / "readnoise.tif"),
    variance=IO.read_tiff(CALIB_DIR / "variance.tif"),
)
smoothing_function = types.SimpleNamespace(args={"sigma": 1.5}, extent=1.5,
                                           smoothing_function=sCMOS.gaussian_filter_stack, data_arg="image")
print(f"FAST_MODE  : {FAST_MODE}")
print(f"Data dir   : {DATA_DIR}")
print(f"Figures dir: {FIG_DIR}")

---
## 2 — Download and fit

Downloads each field's TIFFs and metadata (complete files are skipped) and the stage log, then fits.
Fits are reused only if the `fit_record.json` beside them names the same settings, including the
fitter's stopping rule (for these 40 px boxes: ftol and xtol 10⁻⁸ and 10× the default evaluation limit,
`ImageAnalysisFunctions._lm_kwargs`; looser tolerances stop the fit before it converges and bias the
fingerprint). `REFIT = True` refits anyway; old fits are moved to `old_fits/`. ~10 min per field.

In [ ]:
downloaded = bsd.download_acquisition(zd.REMOTE_FOLDER, DATA_DIR, stem=zd.FIRST_FIELD if FAST_MODE else None)
DATA_DIR = downloaded["images"][0].parent
zlog_file = DATA_DIR / "zlog.csv"   # stage log: measured z-drive position of every plane
if not zlog_file.exists():
    zlog_file = bsd.download_file(f"{zd.REMOTE_FOLDER}/zlog.csv", DATA_DIR)
zlog = pd.read_csv(zlog_file)
fit_files = zd.fit_fields(SupRes_F, DATA_DIR, calib, smoothing_function, refit=REFIT)
print(f"{len(fit_files)} field(s): {[f.name for f in fit_files]}")

---
## 3 — Beads, focus and precision against z

In [ ]:
fits = zd.load_fits(DATA_DIR, IO)
beads, counts = zd.select_beads(fits, SM_E)
for k, v in counts.items():
    print(f"{k:<35} {v:>9,}")

focus_idx, focus_nm, mean_sigma = zd.global_focus(beads)
beads["z_rel_nm"] = beads["z_nm"] - focus_nm
FWHM = 2 * np.sqrt(2 * np.log(2)) * zd.PIXEL_NM
print(f"\nFocus (narrowest mean PSF): z-step {focus_idx}, nominal {focus_nm:+.0f} nm; "
      f"mean FWHM there {mean_sigma[focus_idx] * FWHM:.0f} nm")

# measured stage positions (zlog.csv) relative to the focus step, for the fields analysed
z_meas = (zlog[zlog["posName"].isin([f.split("MMStack_")[-1] for f in beads["field"].unique()])]
          .groupby("plane")["zdrive_rel_nm"].mean())
fwhm = pd.DataFrame({"nominal z / nm": zd.Z_NM - focus_nm,
                     "measured z (zlog) / nm": (z_meas - z_meas[focus_idx]).reindex(range(zd.N_Z_STEPS)).to_numpy(),
                     "mean PSF FWHM / nm": (mean_sigma * FWHM).reindex(range(zd.N_Z_STEPS)).to_numpy()})
print(fwhm.round(0).to_string(index=False))

prec = zd.precision_vs_z(beads)
prec.to_csv(FIG_DIR / "precision_vs_z.csv", index=False)
i0 = prec["z_rel_nm"].abs().idxmin()
print(f"\nAt focus: sigma_xy {prec.loc[i0, 'sigma_xy_nm']:.1f} nm; sigma_colour "
      + ", ".join(f"{zd.PIXEL_NAMES[c]} {prec.loc[i0, 'sigma_colour_' + c]:.4f}" for c in zd.CHANNELS))
for z in (-1000, 900):
    j = (prec["z_rel_nm"] - z).abs().idxmin()
    print(f"At {prec.loc[j, 'z_rel_nm']:+.0f} nm: sigma_xy x{prec.loc[j, 'sigma_xy_nm'] / prec.loc[i0, 'sigma_xy_nm']:.1f}; "
          + ", ".join(f"{zd.PIXEL_NAMES[c]} x{prec.loc[j, 'sigma_colour_' + c] / prec.loc[i0, 'sigma_colour_' + c]:.1f}"
                      for c in zd.CHANNELS))
prec.round(4)

---
## 4 — Figure: precision against z

In [ ]:
fig, axs = plotter.two_column_plot(ncols=2, width_ratios=[1, 1])
zd.plot_precision(axs, prec)
plt.savefig(FIG_DIR / "precision_vs_z.svg", dpi=600, format="svg", bbox_inches="tight")
plt.show()

---
## 5 — Figure: exemplar beads at five distances from focus

One column per bead (beads of the first field, `MOL_INDICES` into their sorted list), one row per z
relative to focus. Each panel is one frame (`FRAME_IDX` of the 100 at that step), cropped around the
bead's mean fitted position, in photoelectrons ((raw − offset) / gain / rQE), each with its own
colour bar; the Bayer key shows the filters at the crop's position on the sensor.

In [ ]:
MOL_INDICES = [0, 1, 2, 3, 4, 5]
Z_POSITIONS = [-1000, -500, 0, 500, 900]   # nm relative to focus (the scan ends at +900)
CROP_SIZE   = 36
FRAME_IDX   = 5
BAYER_KEY_N = 6

field = sorted(beads["field"].unique())[0]
group = zd.field_groups(DATA_DIR)[field]
d = beads[beads["field"] == field]
uids = sorted(d["bead"].unique())
print(f"{field}: {len(uids)} beads; showing {[uids[i] for i in MOL_INDICES if i < len(uids)]}")
centres = d.groupby("bead")[["xc", "yc"]].mean()

roi_x0, roi_y0, roi_w, roi_h = (int(v) for v in IO.metadata_reader_imageJ(group[0][:-len(".ome.tif")] + "_metadata.txt"))
cal = {k: calib[k][roi_y0:roi_y0 + roi_h, roi_x0:roi_x0 + roi_w] for k in ("gain_map", "offset_map", "rqe")}
masks = MaskFunctions.Mask_Functions().get_masks(*calib["gain_map"].shape, get_camera_config("ximea").mosaic_unit)

steps = [focus_idx + int(round(z / zd.Z_STEP_NM)) for z in Z_POSITIONS]
if min(steps) < 0 or max(steps) >= zd.N_Z_STEPS:
    raise ValueError(f"Z_POSITIONS reach outside the scan with focus at step {focus_idx}")
frames = zd.read_frames(group, [s * zd.N_REPEATS + FRAME_IDX for s in steps])
height, width = frames.shape[1:]
half = CROP_SIZE // 2

fig, axs = plotter.two_column_plot(nrows=len(Z_POSITIONS), ncols=len(MOL_INDICES),
                                   width_ratios=np.ones(len(MOL_INDICES)), height_ratios=np.ones(len(Z_POSITIONS)),
                                   height=4.5)
for col, i in enumerate(MOL_INDICES):
    xc, yc = centres.loc[uids[i]]
    x0 = int(np.clip(int(round(xc)) - half, 0, width - CROP_SIZE))
    y0 = int(np.clip(int(round(yc)) - half, 0, height - CROP_SIZE))
    win = (slice(y0, y0 + CROP_SIZE), slice(x0, x0 + CROP_SIZE))
    for row, z in enumerate(Z_POSITIONS):
        ax = axs[row, col]
        crop = (frames[row][win] - cal["offset_map"][win]) / cal["gain_map"][win] / cal["rqe"][win]
        ax = plotter.image_plot(ax, crop, scalebar=True, scalebarsize=300, scalebarlabel="300 nm",
                                vmin=np.percentile(crop, 0.1), vmax=np.percentile(crop, 99.9), colorbar=True,
                                colorbar_label="photoelectrons" if col == len(MOL_INDICES) - 1 else "")[0]
        PlottingBase.add_bayer_key(ax, masks, n=BAYER_KEY_N, row_min=roi_y0 + y0, col_min=roi_x0 + x0)
        if col == 0:
            ax.text(-0.25, 0.5, f"z={z:+.0f} nm", transform=ax.transAxes, fontsize=7, ha="right", va="center")
plt.savefig(FIG_DIR / "exemplar_psfs_grid_488.svg", dpi=300, format="svg")
plt.show()

---
## 6 — Figures: relative pixel QE against z

Kernel density of each pixel type's relative pixel QE, one row per z relative to focus, every curve
scaled to the same peak height; dashed: the mean at focus. Near focus (±200 nm) and over the full scan.

In [ ]:
for name, z_positions, height in [("ridge_colour_qe_vs_z_488_small", [-200, -100, 0, 100, 200], 4.0),
                                  ("ridge_colour_qe_vs_z_488", [-1000, -500, 0, 500, 900], 4.0)]:
    fig = plt.figure(figsize=(4.8, height))
    zd.plot_ridge(fig, beads, z_positions)
    fig.savefig(FIG_DIR / f"{name}.svg", dpi=300, format="svg", bbox_inches="tight")
    plt.show()
    rows = {}
    for z in z_positions:
        sub = beads[np.isclose(beads["z_rel_nm"], beads["z_rel_nm"].unique()[np.argmin(np.abs(beads["z_rel_nm"].unique() - z))])]
        rows[f"{z:+d} nm"] = {zd.PIXEL_NAMES[c]: sub["A_" + c].mean() for c in zd.CHANNELS}
    print(name, "- mean relative pixel QE:")
    print(pd.DataFrame(rows).T.round(3).to_string())